In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('KNNAlgorithmDataset.csv')
df.shape


(569, 33)

In [ ]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [5]:
from sklearn.model_selection import train_test_split
X =df.drop(['diagnosis'],axis=1)
keep_cols = X.columns[:6]
X.drop(columns=X.columns.difference(keep_cols), inplace=True)
Y = df['diagnosis']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [8]:
import math
def fRange(pt1, pt2):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(pt1, pt2)))



from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import math

# 1. Расстояние и классификация для одной точки
def predict_one(test_point, X_train, y_train, k=3):
    # Находим расстояния до всех точек train
    distances = [
        (math.dist(test_point, train_point), label)
        for train_point, label in zip(X_train, y_train)
    ]
    # Берем k ближайших и считаем самый частый класс
    k_nearest = [label for _, label in sorted(distances)[:k]]
    return Counter(k_nearest).most_common(1)[0][0]

# 2. Параллельное предсказание для всего X_test
def predict_parallel(X_test, X_train, y_train, k=3):
    with ProcessPoolExecutor() as executor:
        # executor.map автоматически параллелит функцию по элементам X_test
        results = executor.map(
            predict_one, 
            X_test, 
            [X_train] * len(X_test), 
            [y_train] * len(X_test), 
            [k] * len(X_test)
        )
    return list(results)


# --- Пример запуска ---
if __name__ == '__main__':

    predictions = predict_parallel(X_test, X_train, Y_train, k=3)
    print("Результат:", predictions)


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.